#### Analyze crawler output for RA

In [ ]:
%load_ext autoreload
%autoreload 2


import numpy as np
import pandas as pd
import torch
import pydicom
import matplotlib.pyplot as plt

from pathlib import Path

# MONAI imports
import monai
from monai.data import Dataset, CacheDataset, DataLoader, PILReader
from monai.transforms import (
    LoadImage, LoadImaged, Resized, Compose, SaveImage, 
    Spacingd, SpatialCropd, ResizeWithPadOrCropd
)

import numpy as np
from monai.transforms import (
    Compose,
    LoadImaged,
    Transposed,
    NormalizeIntensityd,
    MapTransform,
#    ScaleIntensityRangePercentiled,
    ScaleIntensityRangePercentilesd
)


# Enable autoreloading
# import importlib


import ra_utils
import ra_utils.data.data_utils
from  ra_utils.data.data_utils import (
    extract_extras_from_filename, 
    extract_extras_from_abspath
)

import ra_utils.visualization.plot_landmarks
import ra_utils.data
import ra_utils.data.data_handler
import ra_utils.data.dataloader_CR_landmarks




dataHandler = ra_utils.data.data_handler.DataHandler_CR_autoscoRA()
dataHandler.load_everything()


In [ ]:
    #Resized(keys=["image"], spatial_size=(512,512))
    #Spacingd(keys=["image"], pixdim=(, 1.0e-2), mode="bilinear"),
    #ResizeWithPadOrCropd(keys=["image"], spatial_size=(512, 512))  # Adjust spatial_size as required


df = dataHandler.df_images_and_landmarks_H
landmarks = ra_utils.data.data_utils.extract_landmarks_from_df(df, image_idx=0)

files_with_extras = [{"image": str(row["image"]), 
                      "landmarks": np.array(ra_utils.data.data_utils.extract_landmarks_from_df(df, image_idx=image_idx), dtype=np.uint16)} for image_idx, row in df.iterrows()]



transform = Compose([
    LoadImaged(keys=["image"], ensure_channel_first=True, reader="PydicomReader"),
    #
    Transposed(keys=["image"], indices=(0, 2, 1)),
    #
    NormalizeIntensityd(
        keys=["image"], 
        nonzero=False,      # whether to exclude zero values in mean/std calc
        channel_wise=True   # compute mean/std for each channel
    ),
     #
    # ScaleIntensityRangePercentilesd(
    #     keys=["image"],
    #     lower=0.05,
    #     upper=99.5,
    #     b_min=0.0,
    #     b_max=1.0,
    #     clip=True,        # clip values outside [b_min, b_max] after scaling
    #     channel_wise=False
    # ),
])


dataset = Dataset(files_with_extras, transform=transform)
dataloader = DataLoader(dataset, batch_size=1, num_workers=2)   # currently only batchsize 1 workes because of different image sizes. However the landmarks correspond to coordinats in pixles, so we need to be carefull if we resize

X = next(iter(dataloader))


# TODO plot X["image"]





In [ ]:
import ra_utils.visualization.plot_landmarks #.plot_landmarks

ra_utils.visualization.plot_landmarks.plot_landmarks(X["image"][0,0,...], X["landmarks"][0,...].numpy());

In [ ]:
plt.hist(X["image"].numpy().ravel(), log=True, bins=50)